# 🧠 Pensamiento Analítico para la Práctica Docente

**Módulo 01** | **Sesión 2** | **Duración estimada: 1h 30min**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-01-fundamentos/notebooks/01_05_pensamiento_analitico_docente.ipynb)

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Descripción | Nivel |
|---|-------------|-------------|-------|
| 1 | Pensamiento basado en datos | Comprender qué significa tomar decisiones informadas por datos en el contexto universitario | Conocimiento |
| 2 | Ciclo analítico | Aplicar el ciclo Pregunta → Datos → Análisis → Insight → Acción a problemas reales de FACES | Aplicación |
| 3 | Preguntas analíticas | Formular preguntas descriptivas, diagnósticas, predictivas y prescriptivas relevantes para la docencia | Aplicación |
| 4 | Integración en la docencia | Diseñar actividades de aula que incorporen análisis de datos reales | Creación |

## 💡 ¿Por qué es importante para tu práctica docente?

Cada semestre tomas decenas de decisiones que impactan a tus estudiantes: cómo estructurar las evaluaciones, qué contenidos priorizar, cómo distribuir las horas de clase, a quiénes ofrecer apoyo adicional. Tradicionalmente, estas decisiones se basan en experiencia e intuición — y eso tiene un valor enorme. Pero, ¿qué pasaría si además pudieras **respaldar tu experiencia con datos**?

El pensamiento analítico no reemplaza tu juicio profesional: lo **potencia**. Te permite:

- **Detectar patrones** que no se ven a simple vista (por ejemplo, que los estudiantes de turno nocturno tienen más deserción)
- **Fundamentar propuestas** ante el consejo de facultad con evidencia concreta
- **Evaluar el impacto** de cambios metodológicos en el rendimiento
- **Enseñar con datos reales** y preparar profesionales más competentes

En esta sesión aprenderemos a pensar analíticamente y a aplicar ese pensamiento tanto en la gestión académica como en el aula.

---

## 📦 Configuración Inicial

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

# Configuración visual
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

# Paleta de colores institucional
COLORES = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
sns.set_palette(COLORES)

print('✅ Librerías cargadas correctamente')

In [ ]:
# Cargar los datasets que usaremos en esta sesión
matricula = pd.read_csv('../../datasets/universidad/matricula_faces.csv')
rendimiento = pd.read_csv('../../datasets/universidad/rendimiento_academico.csv')
presupuesto = pd.read_csv('../../datasets/universidad/presupuesto_facultad.csv')

print('Datasets cargados:')
print(f'  📊 Matrícula FACES:        {matricula.shape[0]:,} registros, {matricula.shape[1]} columnas')
print(f'  📊 Rendimiento académico: {rendimiento.shape[0]:,} registros, {rendimiento.shape[1]} columnas')
print(f'  📊 Presupuesto facultad:  {presupuesto.shape[0]:,} registros, {presupuesto.shape[1]} columnas')

# Vista rápida de cada dataset
print('\n--- Matrícula (primeras 3 filas) ---')
print(matricula.head(3).to_string(index=False))
print('\n--- Rendimiento (primeras 3 filas) ---')
print(rendimiento.head(3).to_string(index=False))
print('\n--- Presupuesto (primeras 3 filas) ---')
print(presupuesto.head(3).to_string(index=False))

---

## 📖 Sección 1: ¿Qué es el pensamiento basado en datos?

### Alfabetización de datos (Data Literacy)

La **alfabetización de datos** es la capacidad de leer, trabajar con, analizar y argumentar con datos. No se trata de ser programador ni estadístico — se trata de poder:

1. **Leer datos**: Entender qué representan los números en una tabla o gráfico
2. **Trabajar con datos**: Saber dónde buscar datos y cómo organizarlos
3. **Analizar datos**: Extraer patrones, tendencias y anomalías
4. **Argumentar con datos**: Usar evidencia cuantitativa para respaldar decisiones

### La mentalidad analítica

La mentalidad analítica consiste en hacerse **preguntas** antes de aceptar conclusiones:

| En vez de pensar... | Piensa analíticamente... |
|---------------------|-------------------------|
| "Los estudiantes de hoy son más flojos" | "¿Ha bajado el promedio? ¿En qué carreras? ¿Desde cuándo?" |
| "La matrícula está cayendo" | "¿En cuánto ha cambiado? ¿En todas las carreras por igual?" |
| "Las becas no sirven" | "¿Cuál es la tasa de aprobación de becados vs. no becados?" |
| "Necesitamos más presupuesto" | "¿En qué partidas hay mayor brecha entre presupuestado y ejecutado?" |

### Decisiones informadas por datos vs. dirigidas por datos

Es importante distinguir:

- **Data-driven** (dirigido por datos): Los datos dictan la decisión. Ejemplo: un algoritmo asigna becas automáticamente según el promedio.
- **Data-informed** (informado por datos): Los datos **complementan** el juicio humano. Ejemplo: un comité revisa los promedios Y las circunstancias personales para asignar becas.

> **En educación, casi siempre queremos ser data-informed, no data-driven.** Los datos son una herramienta, no un sustituto del juicio pedagógico.

In [ ]:
# Ejemplo: del prejuicio al dato
# Prejuicio común: "Los estudiantes que trabajan rinden menos"
# Veamos qué dicen los datos

print('PREGUNTA: ¿Los estudiantes que trabajan tienen peor rendimiento?')
print('=' * 65)

# Calcular estadísticas por grupo
grupo_trabajo = rendimiento.groupby('trabaja').agg(
    n_estudiantes=('promedio', 'count'),
    promedio_medio=('promedio', 'mean'),
    promedio_mediana=('promedio', 'median'),
    materias_aprobadas=('materias_aprobadas', 'mean'),
    asistencia=('asistencia_pct', 'mean')
).round(2)

grupo_trabajo.index = ['No trabaja', 'Sí trabaja']
print(f'\n{grupo_trabajo.to_string()}')

diferencia = grupo_trabajo.loc['No trabaja', 'promedio_medio'] - grupo_trabajo.loc['Sí trabaja', 'promedio_medio']
print(f'\nDiferencia en promedio: {abs(diferencia):.2f} puntos', end='')
if diferencia > 0:
    print(' (a favor de quienes NO trabajan)')
else:
    print(' (a favor de quienes SÍ trabajan)')

print(f'\n💡 El dato nos permite ir más allá del prejuicio y cuantificar la diferencia.')
print(f'   Pero ojo: correlación no es causalidad. Quizás los que trabajan')
print(f'   son también los de mayor edad y más motivación.')

---

## 🔄 Sección 2: El ciclo analítico en el contexto universitario

Todo análisis de datos sigue un ciclo de cinco pasos:

```
🤔 PREGUNTA → 📊 DATOS → 🔍 ANÁLISIS → 💡 INSIGHT → 🎯 ACCIÓN
     ↑                                                    │
     └────────────────────────────────────────────────┘
```

Vamos a recorrer este ciclo completo con una pregunta real de FACES:

> **Pregunta:** *¿Ha cambiado la composición por género en FACES a lo largo de los años?*

### Paso 1: Formular la pregunta

Una buena pregunta analítica es:
- **Específica**: No "cómo van los estudiantes" sino "cuál es la proporción de género por período"
- **Medible**: Debe poder responderse con datos que tenemos o podemos conseguir
- **Relevante**: Debe importarle a alguien (en este caso, a la facultad para políticas de inclusión)

In [ ]:
# Paso 2: Obtener y preparar los datos
print('PASO 2: DATOS')
print('=' * 50)
print(f'\nDataset: matricula_faces.csv')
print(f'Períodos disponibles: {matricula["periodo"].nunique()} (de {matricula["periodo"].min()} a {matricula["periodo"].max()})')
print(f'Variables relevantes: periodo, genero')
print(f'\nConteo por género:')
print(matricula['genero'].value_counts().to_string())

# Preparar: calcular proporción de mujeres por período
genero_por_periodo = matricula.groupby('periodo')['genero'].value_counts(normalize=True).unstack(fill_value=0)
genero_por_periodo = genero_por_periodo * 100  # Convertir a porcentaje
genero_por_periodo = genero_por_periodo.reindex(sorted(genero_por_periodo.index))  # Ordenar por período

print(f'\nProporción por género (%) - primeros 6 períodos:')
print(genero_por_periodo.head(6).round(1).to_string())

In [ ]:
# Paso 3: Analizar - visualizar la tendencia
print('PASO 3: ANÁLISIS')
print('=' * 50)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Evolución temporal de la proporción por género
periodos = genero_por_periodo.index
axes[0].plot(periodos, genero_por_periodo['F'], marker='o', linewidth=2, 
             color=COLORES[1], label='Femenino', markersize=5)
axes[0].plot(periodos, genero_por_periodo['M'], marker='s', linewidth=2, 
             color=COLORES[0], label='Masculino', markersize=5)
axes[0].axhline(y=50, color='gray', linestyle=':', alpha=0.5, label='Paridad (50%)')
axes[0].set_title('Proporción por género en FACES\n(todos los períodos)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Período')
axes[0].set_ylabel('Porcentaje (%)')
axes[0].tick_params(axis='x', rotation=90)
axes[0].legend(fontsize=11)
axes[0].set_ylim(0, 100)

# Gráfico 2: Composición por carrera y género
genero_carrera = matricula.groupby(['carrera', 'genero']).size().unstack(fill_value=0)
genero_carrera_pct = genero_carrera.div(genero_carrera.sum(axis=1), axis=0) * 100

genero_carrera_pct.plot(kind='barh', stacked=True, ax=axes[1], 
                        color=[COLORES[1], COLORES[0]], edgecolor='white', linewidth=1)
axes[1].set_title('Composición por género por carrera\n(todos los períodos)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Porcentaje (%)')
axes[1].set_ylabel('')
axes[1].legend(['Femenino', 'Masculino'], fontsize=11)
axes[1].axvline(x=50, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Paso 4: Extraer el insight
print('PASO 4: INSIGHT')
print('=' * 50)

# Calcular tendencia general
primer_periodo = genero_por_periodo.iloc[0]
ultimo_periodo = genero_por_periodo.iloc[-1]

print(f'\nEn {genero_por_periodo.index[0]}:')
print(f'  Femenino: {primer_periodo["F"]:.1f}%')
print(f'  Masculino: {primer_periodo["M"]:.1f}%')

print(f'\nEn {genero_por_periodo.index[-1]}:')
print(f'  Femenino: {ultimo_periodo["F"]:.1f}%')
print(f'  Masculino: {ultimo_periodo["M"]:.1f}%')

cambio_f = ultimo_periodo['F'] - primer_periodo['F']
print(f'\nCambio en participación femenina: {"+" if cambio_f > 0 else ""}{cambio_f:.1f} puntos porcentuales')

# ¿Qué carrera tiene mayor brecha?
print(f'\nBrecha de género por carrera (% Femenino - % Masculino):')
for carrera in genero_carrera_pct.index:
    brecha = genero_carrera_pct.loc[carrera, 'F'] - genero_carrera_pct.loc[carrera, 'M']
    direccion = 'F > M' if brecha > 0 else 'M > F'
    print(f'  {carrera:25s}: {abs(brecha):.1f} pp ({direccion})')

In [ ]:
# Paso 5: Proponer acciones
print('PASO 5: ACCIÓN')
print('=' * 50)
print()
print('Basados en los datos, posibles acciones para la facultad:')
print()
print('1. 📊 MONITOREAR la tendencia de género semestralmente')
print('   (automatizar un reporte con este mismo código)')
print()
print('2. 🔍 INVESTIGAR las causas de las diferencias entre carreras')
print('   (¿es la oferta académica? ¿el mercado laboral? ¿la cultura?)')
print()
print('3. 🎯 DISEÑAR estrategias de inclusión en carreras')
print('   con mayor desbalance de género')
print()
print('4. 📖 INTEGRAR este tipo de análisis en la docencia')
print('   (que los estudiantes mismos analicen estos datos)')
print()
print('💡 Nota: El ciclo no termina aquí. Las acciones generan')
print('   nuevos datos que nos llevan a nuevas preguntas.')

---

## ❓ Sección 3: Identificar preguntas analíticas relevantes

No todas las preguntas son iguales. Existe una **taxonomía de preguntas analíticas** que nos ayuda a estructurar nuestro pensamiento:

| Tipo | Pregunta clave | Complejidad | Ejemplo en FACES |
|------|---------------|-------------|-------------------|
| **Descriptiva** | ¿Qué ocurrió? | Baja | ¿Cuál es la tasa de deserción por carrera? |
| **Diagnóstica** | ¿Por qué ocurrió? | Media | ¿Por qué bajó la matrícula en 2018? |
| **Predictiva** | ¿Qué podría ocurrir? | Alta | ¿Qué estudiantes tienen mayor riesgo de reprobación? |
| **Prescriptiva** | ¿Qué deberíamos hacer? | Muy alta | ¿Cómo optimizar la asignación de becas? |

Cada tipo construye sobre el anterior. No puedes diagnosticar sin describir primero, ni predecir sin entender las causas.

Veamos un ejemplo de cada tipo con datos reales de FACES.

In [ ]:
# PREGUNTA DESCRIPTIVA: ¿Cuál es la tasa de deserción por carrera?
print('📊 PREGUNTA DESCRIPTIVA')
print('   ¿Cuál es la tasa de retiro (deserción) por carrera?')
print('=' * 55)

# Calcular tasa de retiro por carrera
tasa_retiro = matricula.groupby('carrera')['estatus'].apply(
    lambda x: (x == 'Retirado').sum() / len(x) * 100
).round(2)

print(f'\nTasa de retiro por carrera:')
for carrera, tasa in tasa_retiro.sort_values(ascending=False).items():
    barra = '█' * int(tasa)
    print(f'  {carrera:25s} {tasa:5.1f}% {barra}')

# Visualizar
fig, ax = plt.subplots(figsize=(10, 5))
tasa_retiro_sorted = tasa_retiro.sort_values(ascending=True)
barras = ax.barh(tasa_retiro_sorted.index, tasa_retiro_sorted.values, 
                 color=COLORES[:4], edgecolor='white', height=0.6)
ax.set_xlabel('Tasa de retiro (%)')
ax.set_title('Tasa de retiro por carrera en FACES', fontsize=14, fontweight='bold')

# Agregar etiquetas
for barra in barras:
    ancho = barra.get_width()
    ax.text(ancho + 0.3, barra.get_y() + barra.get_height()/2,
            f'{ancho:.1f}%', va='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

print('\n💡 Este análisis responde al QUÉ: nos dice la situación actual.')
print('   Pero no nos dice POR QUÉ unas carreras tienen más retiro que otras.')

In [ ]:
# PREGUNTA DIAGNÓSTICA: ¿Por qué bajó la matrícula en ciertos períodos?
print('🔍 PREGUNTA DIAGNÓSTICA')
print('   ¿Por qué cambió la matrícula entre períodos?')
print('=' * 55)

# Calcular matrícula total por período
matricula_periodo = matricula.groupby('periodo').size().reindex(sorted(matricula['periodo'].unique()))

# Calcular cambio porcentual entre períodos
cambio_pct = matricula_periodo.pct_change() * 100

# Encontrar los períodos con mayor caída
print(f'\nPeríodos con mayor caída en matrícula:')
mayores_caidas = cambio_pct.nsmallest(5)
for periodo, cambio in mayores_caidas.items():
    print(f'  {periodo}: {cambio:+.1f}%')

# Desglosar por carrera para diagnosticar
print(f'\n¿La caída fue igual en todas las carreras?')
print(f'Desglose del período con mayor caída ({mayores_caidas.index[0]}):')

periodo_peor = mayores_caidas.index[0]
periodos_lista = sorted(matricula['periodo'].unique())
idx_peor = periodos_lista.index(periodo_peor)
periodo_anterior = periodos_lista[idx_peor - 1] if idx_peor > 0 else None

if periodo_anterior:
    mat_antes = matricula[matricula['periodo'] == periodo_anterior].groupby('carrera').size()
    mat_despues = matricula[matricula['periodo'] == periodo_peor].groupby('carrera').size()
    comparacion = pd.DataFrame({
        'Antes': mat_antes,
        'Después': mat_despues
    }).fillna(0).astype(int)
    comparacion['Cambio'] = comparacion['Después'] - comparacion['Antes']
    comparacion['Cambio %'] = ((comparacion['Después'] / comparacion['Antes']) - 1) * 100
    print(f'\n  Comparación {periodo_anterior} vs {periodo_peor}:')
    print(comparacion.round(1).to_string())

print(f'\n💡 El diagnóstico nos ayuda a entender las CAUSAS.')
print(f'   Observar si el cambio es generalizado o concentrado en una')
print(f'   carrera/sede específica nos da pistas sobre la causa.')

In [ ]:
# PREGUNTA PREDICTIVA: ¿Qué estudiantes tienen mayor riesgo de reprobación?
print('🔮 PREGUNTA PREDICTIVA')
print('   ¿Qué características predicen el riesgo de reprobar?')
print('=' * 55)

# Definir "en riesgo": estudiantes que reprueban más de la mitad de sus materias
rendimiento['en_riesgo'] = rendimiento['materias_reprobadas'] > (rendimiento['materias_inscritas'] / 2)

tasa_riesgo_total = rendimiento['en_riesgo'].mean() * 100
print(f'\nTasa general de estudiantes en riesgo: {tasa_riesgo_total:.1f}%')

# Analizar factores de riesgo
factores = {}
print(f'\nTasa de riesgo por factor:')

# Por carrera
print(f'\n  Por carrera:')
for carrera in sorted(rendimiento['carrera'].unique()):
    tasa = rendimiento[rendimiento['carrera'] == carrera]['en_riesgo'].mean() * 100
    print(f'    {carrera:25s}: {tasa:.1f}%')

# Por condición laboral
print(f'\n  Por condición laboral:')
for trabaja, etiqueta in [(True, 'Trabaja'), (False, 'No trabaja')]:
    tasa = rendimiento[rendimiento['trabaja'] == trabaja]['en_riesgo'].mean() * 100
    print(f'    {etiqueta:25s}: {tasa:.1f}%')

# Por beca
print(f'\n  Por beca:')
for beca, etiqueta in [(True, 'Con beca'), (False, 'Sin beca')]:
    tasa = rendimiento[rendimiento['beca'] == beca]['en_riesgo'].mean() * 100
    print(f'    {etiqueta:25s}: {tasa:.1f}%')

# Por asistencia
print(f'\n  Por asistencia:')
for umbral, etiqueta in [(80, 'Alta (≥80%)'), (60, 'Media (60-79%)'), (0, 'Baja (<60%)')]:
    if umbral == 80:
        subset = rendimiento[rendimiento['asistencia_pct'] >= 80]
    elif umbral == 60:
        subset = rendimiento[(rendimiento['asistencia_pct'] >= 60) & (rendimiento['asistencia_pct'] < 80)]
    else:
        subset = rendimiento[rendimiento['asistencia_pct'] < 60]
    tasa = subset['en_riesgo'].mean() * 100
    print(f'    {etiqueta:25s}: {tasa:.1f}% (n={len(subset):,})')

print(f'\n💡 Este tipo de análisis permite identificar factores de riesgo')
print(f'   y diseñar intervenciones tempranas para estudiantes vulnerables.')

In [ ]:
# Visualizar los factores de riesgo
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Factor 1: Asistencia vs Promedio (scatter con color por riesgo)
colores_riesgo = rendimiento['en_riesgo'].map({True: COLORES[3], False: COLORES[0]})
axes[0].scatter(rendimiento['asistencia_pct'], rendimiento['promedio'], 
                c=colores_riesgo, alpha=0.3, s=15)
axes[0].set_xlabel('Asistencia (%)')
axes[0].set_ylabel('Promedio')
axes[0].set_title('Asistencia vs Promedio', fontsize=12, fontweight='bold')
# Leyenda manual
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=COLORES[3], markersize=8, label='En riesgo'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=COLORES[0], markersize=8, label='Sin riesgo')
]
axes[0].legend(handles=legend_elements, fontsize=10)

# Factor 2: Tasa de riesgo por semestre
riesgo_semestre = rendimiento.groupby('semestre')['en_riesgo'].mean() * 100
axes[1].bar(riesgo_semestre.index, riesgo_semestre.values, color=COLORES[2], edgecolor='white')
axes[1].set_xlabel('Semestre')
axes[1].set_ylabel('Tasa de riesgo (%)')
axes[1].set_title('Riesgo por semestre', fontsize=12, fontweight='bold')
axes[1].set_xticks(range(1, 11))

# Factor 3: Distribución de promedio por grupo de riesgo
rendimiento.boxplot(column='promedio', by='en_riesgo', ax=axes[2],
                    patch_artist=True,
                    boxprops=dict(facecolor=COLORES[0], alpha=0.7),
                    medianprops=dict(color=COLORES[3], linewidth=2))
axes[2].set_xticklabels(['Sin riesgo', 'En riesgo'])
axes[2].set_xlabel('')
axes[2].set_ylabel('Promedio')
axes[2].set_title('Promedio por grupo de riesgo', fontsize=12, fontweight='bold')
plt.suptitle('')  # Quitar título automático del boxplot

plt.tight_layout()
plt.show()

In [ ]:
# PREGUNTA PRESCRIPTIVA: ¿Cómo optimizar la asignación de becas?
print('📩 PREGUNTA PRESCRIPTIVA')
print('   ¿Cómo deberíamos asignar las becas para maximizar el rendimiento?')
print('=' * 65)

# Comparar rendimiento de becados vs no becados
becados = rendimiento[rendimiento['beca'] == True]
no_becados = rendimiento[rendimiento['beca'] == False]

print(f'\nComparación de rendimiento:')
print(f'  {"":25s} {"Con beca":>12s} {"Sin beca":>12s} {"Diferencia":>12s}')
print(f'  {"-"*61}')

metricas = [
    ('Promedio', 'promedio'),
    ('Materias aprobadas', 'materias_aprobadas'),
    ('Asistencia (%)', 'asistencia_pct'),
    ('Tasa de riesgo (%)', 'en_riesgo')
]

for nombre, col in metricas:
    val_b = becados[col].mean()
    val_nb = no_becados[col].mean()
    if col == 'en_riesgo':
        val_b *= 100
        val_nb *= 100
    diff = val_b - val_nb
    signo = '+' if diff > 0 else ''
    print(f'  {nombre:25s} {val_b:12.2f} {val_nb:12.2f} {signo}{diff:11.2f}')

print(f'\n  Total becados: {len(becados):,} ({len(becados)/len(rendimiento)*100:.1f}%)')
print(f'  Total sin beca: {len(no_becados):,} ({len(no_becados)/len(rendimiento)*100:.1f}%)')

print(f'\n💡 Recomendaciones basadas en los datos:')
print(f'  1. Analizar si las becas están llegando a quienes más las necesitan')
print(f'  2. Considerar la asistencia como criterio complementario al promedio')
print(f'  3. Evaluar el impacto de la beca en la retención (no solo el promedio)')

---

## 🏫 Sección 4: Integrar datos en la docencia

El pensamiento analítico no es solo para la gestión académica — también es una **herramienta pedagógica poderosa**. Aquí van estrategias prácticas:

### Estrategia 1: Usar datos reales en las tareas

En vez de usar ejemplos ficticios del libro de texto, usa datos reales de la universidad o del país. Los estudiantes se involucran más cuando los datos son relevantes para su contexto.

### Estrategia 2: Enseñar evaluación crítica de estadísticas

Presenta gráficos engañosos o estadísticas sacadas de contexto y pide a los estudiantes que identifiquen los problemas.

### Estrategia 3: Rúbricas basadas en datos

Usa los datos históricos de rendimiento para calibrar tus evaluaciones y establecer expectativas realistas.

### Ejemplo: Actividad de clase con datos reales

In [ ]:
# DEMO: Una actividad de clase que podrías usar
# Tema: "Distribución del rendimiento académico en FACES"
# Materia: Estadística I, Introducción a la Economía, Métodos Cuantitativos

print('═' * 65)
print('  ACTIVIDAD DE CLASE: Análisis de rendimiento académico en FACES')
print('═' * 65)
print()
print('Instrucciones para el estudiante:')
print('1. Observa los datos y gráficos a continuación')
print('2. Responde las preguntas usando evidencia de los datos')
print('3. Formula UNA pregunta adicional que te gustaría investigar')
print()

# Crear un resumen visual completo
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribución de promedios
axes[0, 0].hist(rendimiento['promedio'], bins=30, color=COLORES[0], 
                edgecolor='white', alpha=0.8)
axes[0, 0].axvline(rendimiento['promedio'].mean(), color=COLORES[3], 
                    linewidth=2, linestyle='--', label=f'Media: {rendimiento["promedio"].mean():.1f}')
axes[0, 0].axvline(rendimiento['promedio'].median(), color=COLORES[2], 
                    linewidth=2, linestyle=':', label=f'Mediana: {rendimiento["promedio"].median():.1f}')
axes[0, 0].set_title('Distribución de promedios', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Promedio')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].legend(fontsize=10)

# 2. Promedio por carrera
promedio_carrera = rendimiento.groupby('carrera')['promedio'].mean().sort_values()
axes[0, 1].barh(promedio_carrera.index, promedio_carrera.values, 
                color=COLORES[:4], edgecolor='white', height=0.6)
for i, (carrera, val) in enumerate(promedio_carrera.items()):
    axes[0, 1].text(val + 0.1, i, f'{val:.1f}', va='center', fontweight='bold')
axes[0, 1].set_title('Promedio por carrera', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Promedio')

# 3. Relación asistencia-rendimiento
axes[1, 0].scatter(rendimiento['asistencia_pct'], rendimiento['promedio'],
                   alpha=0.2, s=10, color=COLORES[0])
# Línea de tendencia
z = np.polyfit(rendimiento['asistencia_pct'], rendimiento['promedio'], 1)
p = np.poly1d(z)
x_line = np.linspace(rendimiento['asistencia_pct'].min(), rendimiento['asistencia_pct'].max(), 100)
axes[1, 0].plot(x_line, p(x_line), color=COLORES[3], linewidth=2, label='Tendencia')
axes[1, 0].set_title('Asistencia vs Promedio', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Asistencia (%)')
axes[1, 0].set_ylabel('Promedio')
axes[1, 0].legend(fontsize=10)

# 4. Materias aprobadas vs reprobadas por carrera
aprobadas_carrera = rendimiento.groupby('carrera')['materias_aprobadas'].mean()
reprobadas_carrera = rendimiento.groupby('carrera')['materias_reprobadas'].mean()
x_pos = np.arange(len(aprobadas_carrera))
ancho = 0.35
axes[1, 1].bar(x_pos - ancho/2, aprobadas_carrera.values, ancho, 
               label='Aprobadas', color=COLORES[0], edgecolor='white')
axes[1, 1].bar(x_pos + ancho/2, reprobadas_carrera.values, ancho, 
               label='Reprobadas', color=COLORES[3], edgecolor='white')
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(aprobadas_carrera.index, rotation=30, ha='right')
axes[1, 1].set_title('Materias aprobadas vs reprobadas', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Promedio de materias')
axes[1, 1].legend(fontsize=10)

plt.suptitle('Datos de rendimiento académico en FACES-UC', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nPreguntas para el estudiante:')
print('  a) ¿La distribución de promedios es simétrica? ¿Qué implica esto?')
print('  b) ¿Hay diferencias significativas entre carreras? ¿A qué podría deberse?')
print('  c) ¿Qué relación observas entre asistencia y promedio?')
print('  d) Formula UNA pregunta adicional que podrías responder con estos datos.')

---

## 💻 Sección 5: De la hoja de cálculo al notebook

Muchos docentes hacen sus análisis en Excel. Esto funciona, pero tiene limitaciones importantes para el análisis reproducible:

| Aspecto | Excel | Jupyter Notebook |
|---------|-------|------------------|
| **Reproducibilidad** | Difícil: los pasos no quedan documentados | Fácil: el código ES la documentación |
| **Escalabilidad** | Se vuelve lento con muchos datos (>100K filas) | Maneja millones de filas sin problema |
| **Colaboración** | "Me lo mandas por correo" | Control de versiones con Git |
| **Automatización** | Hay que repetir manualmente cada semestre | Se ejecuta el mismo código con datos nuevos |
| **Visualización** | Gráficos estándar | Gráficos personalizados y profesionales |
| **Narrativa** | Datos separados del análisis | Texto, código y gráficos integrados |
| **Curva de aprendizaje** | Baja | Media (pero vale la pena) |

### Ejemplo: Un flujo típico de Excel reproducido en pandas

Supongamos que en Excel haces lo siguiente cada semestre:
1. Abrir el archivo de presupuesto
2. Filtrar por año
3. Calcular el total presupuestado y ejecutado
4. Calcular la tasa de ejecución
5. Hacer un gráfico de barras

Veamos cómo se hace en pandas:

In [ ]:
# Flujo típico de Excel, ahora en pandas
print('FLUJO DE ANÁLISIS PRESUPUESTARIO')
print('(Antes lo hacías en Excel; ahora es reproducible)')
print('=' * 55)

# Paso 1: Los datos ya están cargados (en Excel: abrir archivo)
print(f'\nPaso 1: Datos cargados ({len(presupuesto):,} registros)')
print(f'  Columnas: {list(presupuesto.columns)}')
print(f'  Años disponibles: {sorted(presupuesto["anio"].unique())}')

# Paso 2: Resumen por año y partida
# En Excel: tabla dinámica. En pandas: groupby + agg
resumen_anual = presupuesto.groupby('anio').agg(
    total_presupuestado=('presupuestado', 'sum'),
    total_ejecutado=('ejecutado', 'sum')
).round(2)

# Paso 3: Calcular tasa de ejecución
# En Excel: nueva columna con fórmula. En pandas: operación vectorizada
resumen_anual['tasa_ejecucion'] = (resumen_anual['total_ejecutado'] / 
                                    resumen_anual['total_presupuestado'] * 100).round(1)

resumen_anual['brecha'] = resumen_anual['total_presupuestado'] - resumen_anual['total_ejecutado']

print(f'\nPasos 2-3: Resumen anual con tasa de ejecución')
# Formatear para mejor lectura
resumen_display = resumen_anual.copy()
resumen_display['total_presupuestado'] = resumen_display['total_presupuestado'].apply(lambda x: f'{x:,.0f}')
resumen_display['total_ejecutado'] = resumen_display['total_ejecutado'].apply(lambda x: f'{x:,.0f}')
resumen_display['brecha'] = resumen_display['brecha'].apply(lambda x: f'{x:,.0f}')
resumen_display['tasa_ejecucion'] = resumen_display['tasa_ejecucion'].apply(lambda x: f'{x:.1f}%')
print(resumen_display.to_string())

In [ ]:
# Paso 4: Visualización (en Excel: gráfico de barras)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

anios = resumen_anual.index
x_pos = np.arange(len(anios))
ancho = 0.35

# Gráfico 1: Presupuestado vs Ejecutado
axes[0].bar(x_pos - ancho/2, resumen_anual['total_presupuestado'] / 1_000_000, ancho,
            label='Presupuestado', color=COLORES[0], edgecolor='white')
axes[0].bar(x_pos + ancho/2, resumen_anual['total_ejecutado'] / 1_000_000, ancho,
            label='Ejecutado', color=COLORES[2], edgecolor='white')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(anios, rotation=45)
axes[0].set_ylabel('Monto (millones)')
axes[0].set_title('Presupuesto vs Ejecución por año', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.1f}M'))

# Gráfico 2: Tasa de ejecución
colores_tasa = [COLORES[0] if t >= 90 else COLORES[2] if t >= 80 else COLORES[3] 
                for t in resumen_anual['tasa_ejecucion']]
barras = axes[1].bar(anios.astype(str), resumen_anual['tasa_ejecucion'], 
                     color=colores_tasa, edgecolor='white')
axes[1].axhline(y=90, color='green', linestyle='--', alpha=0.5, label='Meta (90%)')
axes[1].set_ylabel('Tasa de ejecución (%)')
axes[1].set_title('Tasa de ejecución presupuestaria', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=11)

# Etiquetas en las barras
for barra, tasa in zip(barras, resumen_anual['tasa_ejecucion']):
    axes[1].text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 0.3,
                f'{tasa:.0f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n💡 Ventaja del notebook: este análisis se ejecuta en segundos')
print('   cuando lleguen los datos del próximo año. Solo cambia el archivo CSV.')

In [ ]:
# Desglose por partida (algo más difícil de hacer en Excel)
print('DESGLOSE POR PARTIDA PRESUPUESTARIA')
print('=' * 55)

# Calcular ejecución por partida
resumen_partida = presupuesto.groupby('partida').agg(
    total_presupuestado=('presupuestado', 'sum'),
    total_ejecutado=('ejecutado', 'sum')
).round(2)

resumen_partida['tasa_ejecucion'] = (resumen_partida['total_ejecutado'] / 
                                      resumen_partida['total_presupuestado'] * 100).round(1)
resumen_partida['brecha_pct'] = 100 - resumen_partida['tasa_ejecucion']

# Ordenar por tasa de ejecución
resumen_partida = resumen_partida.sort_values('tasa_ejecucion')

print(f'\n{"Partida":15s} {"Presupuestado":>15s} {"Ejecutado":>15s} {"Ejecución":>10s} {"Brecha":>8s}')
print('-' * 65)
for partida, row in resumen_partida.iterrows():
    print(f'{partida:15s} {row["total_presupuestado"]:15,.0f} {row["total_ejecutado"]:15,.0f} '
          f'{row["tasa_ejecucion"]:9.1f}% {row["brecha_pct"]:7.1f}%')

# Evolución temporal por partida
fig, ax = plt.subplots(figsize=(14, 6))

for i, partida in enumerate(presupuesto['partida'].unique()):
    datos_p = presupuesto[presupuesto['partida'] == partida].groupby('anio').agg(
        tasa=('ejecutado', 'sum')
    )
    datos_total = presupuesto[presupuesto['partida'] == partida].groupby('anio').agg(
        presup=('presupuestado', 'sum')
    )
    tasa_anual = (datos_p['tasa'] / datos_total['presup'] * 100)
    ax.plot(tasa_anual.index.astype(str), tasa_anual.values, 
            marker='o', linewidth=2, label=partida, color=COLORES[i % len(COLORES)], markersize=5)

ax.set_xlabel('Año')
ax.set_ylabel('Tasa de ejecución (%)')
ax.set_title('Evolución de la ejecución presupuestaria por partida', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n💡 En Excel necesitarías crear múltiples tablas dinámicas.')
print('   En pandas, un solo groupby() resuelve el cálculo.')

---

## ✏️ Ejercicios

### Ejercicio 1: Formular preguntas analíticas y responder la descriptiva

Usando el dataset `presupuesto_facultad.csv`, debes:

1. **Formular 3 preguntas** analíticas sobre el presupuesto de FACES:
   - Una **descriptiva** (ej. "¿Cuál es la partida con mayor presupuesto?")
   - Una **diagnóstica** (ej. "¿Por qué bajó la ejecución en cierto año?")
   - Una **predictiva** (ej. "¿Qué partida tendrá mayor brecha el próximo año?")

2. **Responder la pregunta descriptiva con código**: calcula, visualiza y extrae una conclusión.

**Pista**: Usa `presupuesto.groupby()` para agregar los datos por la variable que te interese.

In [ ]:
# EJERCICIO 1: Preguntas analíticas sobre presupuesto
# -----------------------------------------------------------

# Escribe tus tres preguntas aquí (como comentarios o print):
# Pregunta descriptiva:
# Tu pregunta aquí

# Pregunta diagnóstica:
# Tu pregunta aquí

# Pregunta predictiva:
# Tu pregunta aquí


# Ahora responde la pregunta descriptiva con código:
# Tu código aquí


### Ejercicio 2: Diseñar una actividad de clase basada en datos

Piensa en una materia que impartes (o podrías impartir). Diseña una actividad de clase que use datos reales para enseñar un concepto de tu área.

**Parte A** (en la celda markdown siguiente): Describe la actividad
- ¿Qué materia y tema?
- ¿Qué pregunta resolverían los estudiantes?
- ¿Qué datos usarían?
- ¿Qué se espera que aprendan?

**Parte B** (en la celda de código): Escribe el código de soporte
- Carga los datos necesarios
- Haz un análisis exploratorio básico
- Crea al menos un gráfico que los estudiantes usarían

**Parte A: Descripción de la actividad**

*(Escribe aquí tu diseño de actividad)*

- **Materia:** 
- **Tema:** 
- **Pregunta para los estudiantes:** 
- **Datos a utilizar:** 
- **Objetivo de aprendizaje:** 

In [ ]:
# EJERCICIO 2 - Parte B: Código de soporte para tu actividad
# -----------------------------------------------------------

# Carga los datos que necesites
# Tu código aquí


# Análisis exploratorio básico
# Tu código aquí


# Gráfico(s) para la actividad
# Tu código aquí


---

## 📋 Resumen

En esta sesión recorrimos los fundamentos del pensamiento analítico aplicado a la práctica docente:

| Concepto | Punto clave |
|----------|-------------|
| **Alfabetización de datos** | Leer, trabajar, analizar y argumentar con datos |
| **Data-informed vs Data-driven** | En educación, los datos complementan (no sustituyen) el juicio profesional |
| **Ciclo analítico** | Pregunta → Datos → Análisis → Insight → Acción (y vuelta a empezar) |
| **Preguntas descriptivas** | ¿Qué pasó? — El punto de partida de todo análisis |
| **Preguntas diagnósticas** | ¿Por qué pasó? — Buscar causas y correlaciones |
| **Preguntas predictivas** | ¿Qué podría pasar? — Identificar patrones para anticipar |
| **Preguntas prescriptivas** | ¿Qué deberíamos hacer? — Traducir datos en decisiones |
| **De Excel al Notebook** | Reproducibilidad, escalabilidad y automatización |
| **Datos en el aula** | Los datos reales motivan más que los ejemplos ficticios |

### Para llevar a tu práctica

- **Mañana**: La próxima vez que tomes una decisión académica, pregúntate: "¿qué me dicen los datos?"
- **Esta semana**: Identifica UN análisis que haces en Excel y reprodúcelo en un notebook
- **Este semestre**: Incorpora al menos UNA actividad con datos reales en tu materia

## 📚 Referencias

1. Davenport, T. H. (2006). *Competing on Analytics*. Harvard Business Review, 84(1), 98-107.

2. Data Literacy Project. (2021). *The Human Impact of Data Literacy*. Qlik. https://thedataliteracyproject.org/

3. Mandinach, E. B., & Gummer, E. S. (2016). *Data Literacy for Educators: Making It Count in Teacher Preparation and Practice*. Teachers College Press.

4. D'Ignazio, C., & Klein, L. F. (2020). *Data Feminism*. MIT Press. https://data-feminism.mitpress.mit.edu/

5. Wickham, H., & Grolemund, G. (2017). *R for Data Science*. O'Reilly Media. (Capítulos sobre el ciclo de análisis de datos, aplicables a cualquier lenguaje).

6. McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. (Referencia técnica para pandas).